# Modelo jerárquico bayesiano — bielección (bloque largo t-2 -> t)

Misma estructura que `ventana_t-1/02_bayes.ipynb`: pooling parcial por
nivel sobre una única variable estandarizada. Sobre
`data/tfi_data/panel_ventanas_bieleccion.csv` en vez de `panel_ventanas.csv`
-- `icg_pendiente_trim` (trayectoria trimestral del bloque largo) en vez
de `icg_pendiente_vc`. Target: `delta_v` sobre el bloque largo.

In [1]:
import pymc as pm
import numpy as np
import pandas as pd
import sys

general_path = "/workspaces/analisis-politica-economia/"
data_path = f"{general_path}data/tfi_data/"
sys.path.insert(0, f"{general_path}/src")
from ml_models.cargar_panel import cargar_panel, cargar_panel_apilado

df_apilado = cargar_panel_apilado(
    justificacion="Modelo jerárquico bayesiano con pooling parcial por nivel (bielección t-2 -> t)",
    panel_path=f"{data_path}panel_ventanas_bieleccion.csv",
)

ModuleNotFoundError: No module named 'pymc'

In [ ]:
VARIABLE = "icg_pendiente_trim"
NIVELES_COD = {"municipal": 0, "provincial": 1, "nacional": 2}
datos_modelo = df_apilado[["nivel", "id_transicion", VARIABLE, "delta_v"]].dropna().reset_index(drop=True)
datos_modelo["nivel_cod"] = datos_modelo["nivel"].map(NIVELES_COD)
media_x = datos_modelo[VARIABLE].mean()
desvio_x = datos_modelo[VARIABLE].std(ddof=0)
datos_modelo["x_std"] = (datos_modelo[VARIABLE] - media_x) / desvio_x
print(datos_modelo.groupby("nivel").size())
print(datos_modelo[["nivel", "x_std", "delta_v"]].head())

In [ ]:
n_niveles = 3
nivel_idx = datos_modelo["nivel_cod"].values
x = datos_modelo["x_std"].values
y = datos_modelo["delta_v"].values

with pm.Model() as modelo_jerarquico:
    alpha_mu = pm.Normal("alpha_mu", mu=0, sigma=10)
    alpha_sigma = pm.HalfNormal("alpha_sigma", sigma=5)
    beta_mu = pm.Normal("beta_mu", mu=0, sigma=5)
    beta_sigma = pm.HalfNormal("beta_sigma", sigma=3)
    # parametrización no-centrada (evita divergencias con pocos grupos)
    z_alpha = pm.Normal("z_alpha", mu=0, sigma=1, shape=n_niveles)
    z_beta = pm.Normal("z_beta", mu=0, sigma=1, shape=n_niveles)
    alpha_nivel = pm.Deterministic("alpha_nivel", alpha_mu + alpha_sigma * z_alpha)
    beta_nivel = pm.Deterministic("beta_nivel", beta_mu + beta_sigma * z_beta)
    sigma = pm.HalfNormal("sigma", sigma=10)
    mu = alpha_nivel[nivel_idx] + beta_nivel[nivel_idx] * x
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y)

In [ ]:
with modelo_jerarquico:
    trace = pm.sample(
        draws=2000,
        tune=2000,
        chains=4,
        target_accept=0.99,
        random_seed=42,
        return_inferencedata=True,
    )
print("Divergencias totales:", trace.sample_stats["diverging"].sum().item())

In [ ]:
import arviz as az
resumen = az.summary(trace, var_names=["alpha_mu", "alpha_sigma", "beta_mu", "beta_sigma", "sigma", "alpha_nivel", "beta_nivel"])
print(resumen)

### Contraste sin pooling

In [ ]:
with pm.Model() as modelo_sin_pooling:
    alpha_nivel = pm.Normal("alpha_nivel", mu=0, sigma=10, shape=n_niveles)
    beta_nivel = pm.Normal("beta_nivel", mu=0, sigma=10, shape=n_niveles)
    sigma = pm.HalfNormal("sigma", sigma=10)
    mu = alpha_nivel[nivel_idx] + beta_nivel[nivel_idx] * x
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y)
    trace_sin_pooling = pm.sample(draws=2000, tune=2000, chains=4, target_accept=0.95, random_seed=42)
print(az.summary(trace_sin_pooling, var_names=["beta_nivel"]))

### LOO-CV bayesiano manual, por nivel

In [ ]:
def loocv_bayesiano_nivel(datos_nivel: pd.DataFrame, draws=1000, tune=1000) -> float:
    """LOO manual: para cada punto del nivel, ajusta el modelo sin él y
    predice con la media posterior de alpha/beta. Devuelve MSE, comparable
    directo con baseline_trivial_loocv (ml_models.lasso)."""
    n = len(datos_nivel)
    errores = []
    for i in range(n):
        train = datos_nivel.drop(datos_nivel.index[i])
        test = datos_nivel.iloc[i]
        with pm.Model():
            alpha = pm.Normal("alpha", mu=0, sigma=10)
            beta = pm.Normal("beta", mu=0, sigma=10)
            sigma = pm.HalfNormal("sigma", sigma=10)
            mu = alpha + beta * train["x_std"].values
            pm.Normal("y_obs", mu=mu, sigma=sigma, observed=train["delta_v"].values)
            tr = pm.sample(draws=draws, tune=tune, chains=2, target_accept=0.95, progressbar=False, random_seed=42)
        pred = tr.posterior["alpha"].mean().item() + tr.posterior["beta"].mean().item() * test["x_std"]
        errores.append((pred - test["delta_v"]) ** 2)
    return np.mean(errores)

In [ ]:
mse_sin_pooling = {}
for niv in ["municipal", "provincial", "nacional"]:
    datos_nivel = datos_modelo[datos_modelo["nivel"] == niv].reset_index(drop=True)
    mse_sin_pooling[niv] = loocv_bayesiano_nivel(datos_nivel)
    print(f"MSE LOO bayesiano ({niv}, sin pooling):", mse_sin_pooling[niv])